# COTREC + CSGNN — RetailRocket (Colab)

**Quan trọng:** Colab chỉ có **1 GPU**. Không chạy COTREC và CSGNN **song song** — tranh VRAM/compute, log đứng ở `[0/7963]` rất lâu, hoặc session disconnect sẽ kill process.

Notebook này chạy **tuần tự**: COTREC xong → CSGNN.

Ước tính (T4, full retailrocket):
- COTREC 30 epoch: ~40–50 giờ
- CSGNN 1 epoch: ~2–3 giờ

In [ ]:
# ===== CONFIG =====
REPO_DIR = "/content/test_all"   # clone/upload repo vào đây
DATASET = "retailrocket"
EPOCH_COTREC = 30
EPOCH_CSGNN = 1
EMB_SIZE_CSGNN = 100
BETA_CSGNN = 0.005

# Smoke test nhanh (1 epoch COTREC, vài phút): đặt True
SMOKE = False
SMOKE_EPOCH_COTREC = 1

In [ ]:
import os, sys, subprocess
from datetime import datetime
from pathlib import Path

repo = Path(REPO_DIR)
assert (repo / "nhom2/COTREC/main.py").exists(), f"Không thấy repo tại {repo}"
os.chdir(repo)
sys.path.insert(0, str(repo))

get_ipython().system("pip install -q scipy")

epoch_cotrec = SMOKE_EPOCH_COTREC if SMOKE else EPOCH_COTREC

def run_one(name: str, script: str, *extra_args: str) -> int:
    ts = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
    log = repo / "Log" / name / DATASET / f"log-{ts}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, "-u", script,
        "--dataset", DATASET,
        *extra_args,
    ]
    print("=" * 60)
    print(f"START {name}  {datetime.now().isoformat()}")
    print("CMD:", " ".join(cmd))
    print("LOG:", log)
    with open(log, "w", buffering=1) as f:
        proc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    print(f"DONE {name}  exit={proc.returncode}  {datetime.now().isoformat()}")
    print("--- tail log ---")
    get_ipython().system(f"tail -30 {log}")
    return proc.returncode

# Tuần tự — KHÔNG chạy song song trên 1 GPU Colab
ec = run_one(
    "COTREC",
    "nhom2/COTREC/main.py",
    "--epoch", str(epoch_cotrec),
)
if ec != 0:
    raise SystemExit(f"COTREC failed with exit={ec}")

es = run_one(
    "CSGNN",
    "nhom2/CSGNN/main.py",
    "--epoch", str(EPOCH_CSGNN),
    "--embSize", str(EMB_SIZE_CSGNN),
    "--beta", str(BETA_CSGNN),
)
if es != 0:
    raise SystemExit(f"CSGNN failed with exit={es}")

print("ALL DONE")